### Silver Pipeline
Cleans, formats, and enriches the bronze tables. The transactions table is enhanced with the MCC description lookup and the fraud label.

Produces three silver tables:
 - `catalog.silver.transactions` (cleaned + enriched)
 - `catalog.silver.mcc_codes` (cleaned)
 - `catalog.silver.fraud_labels` (cleaned)

Run this as the `Silver` task, after `Bronze`.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS catalog.silver;

In [0]:
from pyspark.sql import functions as F

In [0]:
bronze_transactions_df = spark.read.table("catalog.bronze.transactions")
bronze_mcc_codes_df = spark.read.table("catalog.bronze.mcc_codes")
bronze_fraud_labels_df = spark.read.table("catalog.bronze.fraud_labels")

In [0]:
silver_mcc_codes_df = (bronze_mcc_codes_df
    .withColumn("mcc", F.col("mcc").cast("int"))
    .dropDuplicates(["mcc"])
)

silver_fraud_labels_df = (bronze_fraud_labels_df
    .withColumn("is_fraud", F.when(F.col("fraud_flag") == "Yes", True).otherwise(False))
    .select("transaction_id", "is_fraud")
    .dropDuplicates(["transaction_id"])
)

display(silver_mcc_codes_df)
display(silver_fraud_labels_df)

In [0]:
silver_transactions_df = (bronze_transactions_df
    # amount arrives like "$123.45" -> clean numeric
    .withColumn("amount", F.regexp_replace(F.col("amount"), "[$,]", "").cast("double"))
    # parse the timestamp
    .withColumn("txn_timestamp", F.to_timestamp(F.col("date"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("txn_date", F.to_date(F.col("txn_timestamp")))
    # drop obviously broken rows
    .filter(F.col("id").isNotNull() & F.col("txn_timestamp").isNotNull())
    .dropDuplicates(["id"])
)

display(silver_transactions_df)

In [0]:
silver_transactions_enriched_df = (silver_transactions_df
    .join(silver_mcc_codes_df, on="mcc", how="left")
    .join(silver_fraud_labels_df,
          silver_transactions_df["id"] == silver_fraud_labels_df["transaction_id"],
          how="left")
    .drop("transaction_id")
    .withColumn("is_fraud", F.coalesce(F.col("is_fraud"), F.lit(False)))
)

In [0]:
silver_transactions_final_df = (silver_transactions_enriched_df
    .withColumn("day_of_week", F.date_format(F.col("txn_timestamp"), "EEEE"))
    .withColumn("day_of_week_num", F.dayofweek(F.col("txn_timestamp")))   # 1 = Sunday
    .withColumn("year_month", F.date_format(F.col("txn_timestamp"), "yyyy-MM"))
    .withColumn("hour_of_day", F.hour(F.col("txn_timestamp")))
    .withColumn(
        "time_of_day",
        F.when((F.col("hour_of_day") >= 5) & (F.col("hour_of_day") < 12), "Morning")
         .when((F.col("hour_of_day") >= 12) & (F.col("hour_of_day") < 17), "Afternoon")
         .when((F.col("hour_of_day") >= 17) & (F.col("hour_of_day") < 21), "Evening")
         .otherwise("Night")
    )
    .withColumn("week_of_year", F.weekofyear(F.col("txn_timestamp")))
)

display(silver_transactions_final_df)

In [0]:
silver_transactions_final_df.write.mode("overwrite").saveAsTable("catalog.silver.transactions")
silver_mcc_codes_df.write.mode("overwrite").saveAsTable("catalog.silver.mcc_codes")
silver_fraud_labels_df.write.mode("overwrite").saveAsTable("catalog.silver.fraud_labels")